In [0]:
#==============================================================
# Import Required Libraries
#==============================================================

# PySpark SQL Functions
from pyspark.sql import functions as F
from pyspark.sql.functions import *

# Spark SQL Data Types
from pyspark.sql.types import *

# Delta Lake
from delta.tables import DeltaTable

# Date & Time
from datetime import datetime

# UUID
import uuid

**Initializes the notebook configuration**

In [0]:
#==============================================================
# Notebook Configuration
#==============================================================

# Generate execution ID
execution_id = str(uuid.uuid4())

# Catalog Name
catalog_name = "adb_databricks_project_dev"

# Metadata Schema
metadata_schema = "metadata"

# Bronze Schema
bronze_schema = "bronze"

# Silver Schema
silver_schema = "silver"

# Display Configuration
print("Execution ID :", execution_id)
print("Catalog      :", catalog_name)
print("Bronze       :", bronze_schema)
print("Silver       :", silver_schema)

**Read Source Configuration**

In [0]:
#==============================================================
# Read Source Configuration
#==============================================================

# Read active source configurations
source_config_df = spark.sql(f"""

SELECT *

FROM {catalog_name}.{metadata_schema}.source_config

WHERE active_flag = 'Y'

AND source_type = 'FILE'

ORDER BY
entity_name

""")

# Display source configuration
display(source_config_df)

# Convert DataFrame into Python objects
source_config_list = source_config_df.collect()

# Display total entities
print("Total Entities :", len(source_config_list))

**Read Bronze Table**

In [0]:
#==============================================================
# Function Name : read_bronze_table()
#==============================================================

def read_bronze_table(config):

    # Read Bronze table
    df = spark.table(

        f"{config['target_catalog']}."
        f"{config['bronze_schema']}."
        f"{config['bronze_table']}"

    )

    # Display source table
    print("Bronze Table :", config["bronze_table"])

    # Display total records
    print("Total Records :", df.count())

    return df

**Apply Incremental Load**

In [0]:
#==============================================================
# Function Name : apply_incremental_load()
#
# Purpose :
# Reads only new records from Bronze using the configured
# watermark column.
#
# Called From :
# Silver Driver
#
# Layer :
# Silver
#==============================================================

from pyspark.sql.functions import col, max


def apply_incremental_load(df, config):

    # Read watermark column
    watermark_column = config["watermark_column"]

    # Full Load if watermark not configured
    if watermark_column is None or watermark_column == "":

        print("Full Load")
        return df

    # Silver table
    silver_table = (
        f"{config['target_catalog']}."
        f"{config['silver_schema']}."
        f"{config['silver_table']}"
    )

    # First Load
    if not spark.catalog.tableExists(silver_table):

        print("First Load")
        return df

    # Read last watermark
    last_watermark = (
        spark.table(silver_table)
        .agg(
            max(col(watermark_column)).alias("last_watermark")
        )
        .collect()[0]["last_watermark"]
    )

    # Silver Empty
    if last_watermark is None:

        print("Silver Empty")
        return df

    print("Last Watermark :", last_watermark)

    # Read only new records
    df = df.filter(
        col(watermark_column) > last_watermark
    )

    print("Incremental Records :", df.count())

    return df

**Standardize Column Names**

In [0]:
#==============================================================
# Function Name : standardize_column_names()
#==============================================================

def standardize_column_names(df):

    # Standardize column names
    for column in df.columns:

       new_column = (
          column.strip()
          .lower()
          .replace(" ", "_")
          .replace("-", "_")
          .replace("/", "_")
          .replace("(", "")
          .replace(")", "")
          .replace(".", "_")
         )

    df = df.withColumnRenamed(column, new_column)

    # Display updated columns
    print("Column names standardized.")

    return df

**Trim String Columns**

In [0]:
#==============================================================
# Function Name : trim_string_columns()
#==============================================================

def trim_string_columns(df):

    # Get all string columns
    string_columns = [

        field.name

        for field in df.schema.fields

        if isinstance(field.dataType, StringType)

    ]

    # Trim all string columns
    for column in string_columns:

        df = df.withColumn(

            column,

            trim(col(column))

        )

    # Display status
    print("String columns trimmed.")

    return df

**Convert Data Types**

In [0]:
#==============================================================
# Function Name : convert_data_types()
#==============================================================

def convert_data_types(df):

    # Convert date columns
    for column in df.columns:

        if "date" in column.lower():

            df = df.withColumn(

                column,

                to_date(col(column))

            )

    # Convert timestamp columns
    for column in df.columns:

        if "timestamp" in column.lower():

            df = df.withColumn(

                column,

                to_timestamp(col(column))

            )

    # Convert amount columns
    for column in df.columns:

        if "amount" in column.lower():

            df = df.withColumn(

                column,

                col(column).cast("double")

            )

    # Convert price columns
    for column in df.columns:

        if "price" in column.lower():

            df = df.withColumn(

                column,

                col(column).cast("double")

            )

    # Convert quantity columns
    for column in df.columns:

        if "quantity" in column.lower():

            df = df.withColumn(

                column,

                col(column).cast("int")

            )

    # Display status
    print("Data types converted.")

    return df

**Handle Null Values**

In [0]:
#==============================================================
# Function Name : handle_null_values()
#==============================================================

def handle_null_values(df):

    # Replace null values in string columns
    for field in df.schema.fields:

        if isinstance(field.dataType, StringType):

            df = df.withColumn(

                field.name,

                when(
                    col(field.name).isNull(),
                    lit("Unknown")
                ).otherwise(col(field.name))

            )

    # Replace null values in integer columns
    for field in df.schema.fields:

        if isinstance(field.dataType, IntegerType):

            df = df.withColumn(

                field.name,

                when(
                    col(field.name).isNull(),
                    lit(0)
                ).otherwise(col(field.name))

            )

    # Replace null values in long columns
    for field in df.schema.fields:

        if isinstance(field.dataType, LongType):

            df = df.withColumn(

                field.name,

                when(
                    col(field.name).isNull(),
                    lit(0)
                ).otherwise(col(field.name))

            )

    # Replace null values in decimal columns
    for field in df.schema.fields:

        if isinstance(field.dataType, DoubleType):

            df = df.withColumn(

                field.name,

                when(
                    col(field.name).isNull(),
                    lit(0.0)
                ).otherwise(col(field.name))

            )

    # Display status
    print("Null values handled.")

    return df

**Apply Business Rules**

In [0]:
#==============================================================
# Function Name : apply_business_rules()
#==============================================================

def apply_business_rules(df, config):

    # Get primary key
    primary_key = config["primary_key"]

    # Remove records with null primary key
    if primary_key in df.columns:

        df = df.filter(

            col(primary_key).isNotNull()

        )

    # Check amount columns
    for column in df.columns:

        if "amount" in column.lower():

            df = df.filter(

                col(column) >= 0

            )

    # Check price columns
    for column in df.columns:

        if "price" in column.lower():

            df = df.filter(

                col(column) >= 0

            )

    # Check quantity columns
    for column in df.columns:

        if "quantity" in column.lower():

            df = df.filter(

                col(column) >= 0

            )

    # Remove duplicate rows
    df = df.dropDuplicates()

    # Display status
    print("Business rules applied.")

    return df

**Remove Duplicate Records**

In [0]:
#==============================================================
# Function Name : remove_duplicates()
#==============================================================

def remove_duplicates(df, config):

    # Get primary key
    primary_key = config["primary_key"]

    # Remove duplicate records
    if primary_key in df.columns:

        before_count = df.count()

        df = df.dropDuplicates([primary_key])

        after_count = df.count()

        print("Duplicate Records Removed :", before_count - after_count)

    else:

        print(f"Primary Key '{primary_key}' not found.")

    return df

**Add Audit Columns**

In [0]:
#==============================================================
# Function Name : add_audit_columns()
#==============================================================

def add_audit_columns(df):

    # Add record creation timestamp
    df = df.withColumn(

        "created_date",

        current_timestamp()

    )

    # Add record update timestamp
    df = df.withColumn(

        "updated_date",

        current_timestamp()

    )

    # Add execution ID
    df = df.withColumn(

        "execution_id",

        lit(execution_id)

    )

    # Add load date
    df = df.withColumn(

        "load_date",

        current_date()

    )

    # Display status
    print("Audit columns added.")

    return df

** Merge Data into Silver**

In [0]:
#==============================================================
# Function Name : merge_to_silver()
#==============================================================

def merge_to_silver(df, config):

    # Get target Silver table
    silver_table = (

        f"{config['target_catalog']}."
        f"{config['silver_schema']}."
        f"{config['silver_table']}"

    )

    # Get primary key
    primary_key = config["primary_key"]

    # Create Silver table if it does not exist
    if not spark.catalog.tableExists(silver_table):

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(silver_table)

        print("Silver table created :", silver_table)

        return

    # Read Silver table
    delta_table = DeltaTable.forName(

        spark,

        silver_table

    )

    # Merge condition
    merge_condition = (

        f"target.{primary_key} = source.{primary_key}"

    )

    # Merge Bronze into Silver
    (

        delta_table.alias("target")

        .merge(

            df.alias("source"),

            merge_condition

        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()

    )

    print("Merge completed :", silver_table)

**Insert Transformation Configuration**

In [0]:
import uuid

def insert_transformation_config(config, df):

    primary_key = config["primary_key"]

    for index, field in enumerate(df.schema.fields, start=1):

        transformation_id = str(uuid.uuid4())

        source_column = field.name

        target_column = field.name

        target_data_type = field.dataType.simpleString().upper()

        transformation_rule = "DIRECT_MAPPING"

        if "date" in field.name.lower():
            transformation_rule = "TO_DATE"

        elif "amount" in field.name.lower():
            transformation_rule = "DOUBLE_CAST"

        elif "price" in field.name.lower():
            transformation_rule = "DOUBLE_CAST"

        elif isinstance(field.dataType, StringType):
            transformation_rule = "TRIM"

        try:

            spark.sql(f"""

            INSERT INTO {config['target_catalog']}.metadata.transformation_config

            VALUES
            (

                '{transformation_id}',

                '{config["entity_name"]}',

                '{config["bronze_table"]}',

                '{config["silver_table"]}',

                '{source_column}',

                '{target_column}',

                '{target_data_type}',

                '{transformation_rule}',

                '{primary_key}',

                {index},

                'Y',

                'Y'

            )

            """)

            print(f"Inserted : {source_column}")

        except Exception as e:

            print(e)

**Execution Log**

In [0]:
#==============================================================
# Function Name : insert_execution_log()
#
# Purpose :
# Inserts Silver notebook execution details into execution log.
#
# Called From :
# Silver Driver
#
# Layer :
# Metadata
#==============================================================

def insert_execution_log(
    config,
    source_start_time,
    status,
    records_read,
    records_loaded,
    error_message=None
):

    # Silver layer
    layer = "SILVER"

    # Silver works at entity level
    file_format = "AUTO"

    spark.sql(f"""

        INSERT INTO
        {config['target_catalog']}.metadata.execution_log

        VALUES
        (

            '{execution_id}',

            '{escape_sql(config["entity_name"])}',

            '{layer}',

            '{escape_sql("FILE")}',

            '{file_format}',

            '{source_start_time}',

            current_timestamp(),

            '{escape_sql(status)}',

            {records_read},

            {records_loaded},

            '{escape_sql(error_message)}'

        )

    """)

** Escape SQL Text**

In [0]:
#==============================================================
# Function Name : escape_sql()
#==============================================================

def escape_sql(value):

    if value is None:
        return ""

    return str(value).replace("'", "''")

**Driver Cell**

In [0]:
#==============================================================
# Silver Driver
#==============================================================

from datetime import datetime

# Process every entity
for config in source_config_list:

    print("")
    print("===================================================")
    print("Entity :", config["entity_name"])
    print("===================================================")

    # Execution Start Time
    source_start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    status = "SUCCESS"

    error_message = None

    records_read = 0

    records_loaded = 0

    try:

        #------------------------------------------------------
        # Read Bronze Table
        #------------------------------------------------------

        df = read_bronze_table(config)

        #------------------------------------------------------
        # Apply Incremental Load
        #------------------------------------------------------

        df = apply_incremental_load(
            df,
            config
        )

        records_read = df.count()

        #------------------------------------------------------
        # Standardize Column Names
        #------------------------------------------------------

        df = standardize_column_names(df)

        #------------------------------------------------------
        # Trim String Columns
        #------------------------------------------------------

        df = trim_string_columns(df)

        #------------------------------------------------------
        # Convert Data Types
        #------------------------------------------------------

        df = convert_data_types(df)

        #------------------------------------------------------
        # Handle Null Values
        #------------------------------------------------------

        df = handle_null_values(df)

        #------------------------------------------------------
        # Apply Business Rules
        #------------------------------------------------------

        df = apply_business_rules(
            df,
            config
        )

        #------------------------------------------------------
        # Remove Duplicate Records
        #------------------------------------------------------

        df = remove_duplicates(
            df,
            config
        )

        #------------------------------------------------------
        # Add Audit Columns
        #------------------------------------------------------

        df = add_audit_columns(df)

        #------------------------------------------------------
        # Store Transformation Metadata
        #------------------------------------------------------

        insert_transformation_config(
            config,
            df
        )

        #------------------------------------------------------
        # Merge into Silver
        #------------------------------------------------------

        merge_to_silver(
            df,
            config
        )

        records_loaded = df.count()

    except Exception as e:

        status = "FAILED"

        error_message = str(e)

        print(error_message)

    #------------------------------------------------------
    # Insert Execution Log
    #------------------------------------------------------

    insert_execution_log(

        config=config,

        source_start_time=source_start_time,

        status=status,

        records_read=records_read,

        records_loaded=records_loaded,

        error_message=error_message

    )

print("")
print("======================================")
print("Silver Notebook Completed Successfully")
print("======================================")

In [0]:
%sql
SELECT *
FROM adb_databricks_project_dev.metadata.transformation_config;

In [0]:
%sql

SELECT *
FROM adb_databricks_project_dev.metadata.execution_log
ORDER BY end_time DESC;